# 06 - Attitude Providers

This notebook shows the current NSTK attitude workflow:

1. call `nstk.initialize(offline=True)` once
2. build a propagator with `attitude_provider=...` when needed
3. wrap it with `Orbit`
4. query quaternion and body-frame angular-vector outputs
5. swap providers later with `Orbit.set_attitude_provider(...)`


In [1]:
# Ensure local package import when running directly from examples/.
import sys
from pathlib import Path

repo_root = Path.cwd().resolve().parent if Path.cwd().name == "examples" else Path.cwd().resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))


In [2]:
import nstk
import numpy as np
from astropy.time import Time

nstk.initialize(offline=True)

from nstk.propagation import (
    Orbit,
    RateLimitedYawSteeringProvider,
    build_ideal_nadir_sun_constrained_attitude_provider,
    build_two_body_propagator,
)
from org.hipparchus.geometry.euclidean.threed import RotationOrder, Vector3D  # type: ignore
from org.orekit.attitudes import LofOffset  # type: ignore
from org.orekit.frames import LOFType  # type: ignore

np.set_printoptions(precision=6, suppress=True)

epoch = Time("2026-01-01T00:00:00", scale="utc")
t_query = np.array([0.0, 60.0, 120.0], dtype=np.float64)


def make_orbit(attitude_provider=None):
    return Orbit(
        build_two_body_propagator(
            epoch=epoch,
            a=7050e3,
            e=0.002,
            i=np.deg2rad(97.4),
            raan=np.deg2rad(5.0),
            argp=np.deg2rad(45.0),
            anomaly=np.deg2rad(0.0),
            anomaly_type="mean",
            inertial_frame="gcrf",
            attitude_provider=attitude_provider,
        )
    )


## 1) Default VVLH attitude from the propagator factory

NSTK's propagator factories attach a default Orekit `LofOffset(..., LOFType.VVLH)` attitude law when you do not pass an explicit provider.


In [3]:
orbit_default = make_orbit()
q_default = orbit_default.get_attitude_quat(t_query, quaternion_convention="scalar_last")

print("native frame:", orbit_default.native_frame.getName())
print("provider class:", orbit_default.get_attitude_provider().getClass().getSimpleName())
print("first quaternion [q1, q2, q3, q4]:", q_default[0])


native frame: GCRF
provider class: LofOffset
first quaternion [q1, q2, q3, q4]: [ 0.064887 -0.919999 -0.042906  0.384123]


## 2) Quaternion convention and body-frame angular vectors

`Orbit.get_attitude_quat(...)` returns STK-style scalar-last quaternions `[q1, q2, q3, q4]` from the selected reference frame into the spacecraft body frame.

`Orbit.get_attitude_spin(...)` and `Orbit.get_attitude_acceleration(...)` always return body-frame angular vectors in `rad/s` and `rad/s^2`.


In [4]:
state0 = orbit_default.propagator.propagate(orbit_default.propagator.getInitialState().getDate())
rot0 = state0.getAttitude().getRotation()
body_plus_z_in_reference = rot0.applyInverseTo(Vector3D.PLUS_K)
spin = orbit_default.get_attitude_spin(t_query)
accel = orbit_default.get_attitude_acceleration(t_query)

print("body +Z axis in the reference/native frame:", [
    float(body_plus_z_in_reference.getX()),
    float(body_plus_z_in_reference.getY()),
    float(body_plus_z_in_reference.getZ()),
])
print("spin shape:", spin.shape, " first sample [rad/s]:", spin[0])
print("acceleration shape:", accel.shape, " first sample [rad/s^2]:", accel[0])


body +Z axis in the reference/native frame: [-0.7123534950998548, 0.029097265643120228, -0.701217403628229]
spin shape: (3, 3)  first sample [rad/s]: [ 0.       -0.001071  0.      ]
acceleration shape: (3, 3)  first sample [rad/s^2]: [-0.  0. -0.]


## 3) Pass a raw Orekit provider at construction time

After `nstk.initialize(offline=True)`, you can import Orekit classes directly from `org.orekit...` and use them with NSTK factories.


In [5]:
qsw_provider = LofOffset(orbit_default.native_frame, LOFType.QSW)
orbit_qsw = make_orbit(qsw_provider)
q_qsw = orbit_qsw.get_attitude_quat(t_query, quaternion_convention="scalar_last")

print("QSW provider class:", orbit_qsw.get_attitude_provider().getClass().getSimpleName())
print("QSW first quaternion:", q_qsw[0])
print("QSW matches default VVLH:", bool(np.allclose(q_qsw, q_default)))


QSW provider class: LofOffset
QSW first quaternion: [-0.705957  0.256947 -0.278929 -0.598165]
QSW matches default VVLH: False


## 4) Use NSTK's ideal nadir-plus-Sun helper

`build_ideal_nadir_sun_constrained_attitude_provider(...)` returns the unrestricted geometric law. It is a clean way to attach STK-style nadir/Sun phasing without configuring Orekit classes manually.


In [6]:
ideal_sun = build_ideal_nadir_sun_constrained_attitude_provider(inertial_frame="gcrf")
orbit_sun = make_orbit(ideal_sun)
q_sun = orbit_sun.get_attitude_quat(t_query, quaternion_convention="scalar_last")

print("ideal nadir+sun provider class:", orbit_sun.get_attitude_provider().getClass().getSimpleName())
print("ideal nadir+sun first quaternion:", q_sun[0])


ideal nadir+sun provider class: AlignedAndConstrained
ideal nadir+sun first quaternion: [ 0.770707 -0.507643 -0.32999   0.198535]


## 5) Swap the provider directly on `Orbit`

If you already have an `Orbit`, use `Orbit.set_attitude_provider(...)`. It accepts raw Orekit providers and NSTK wrapper providers and updates the underlying bridge without requiring a manual cache clear.


In [7]:
orbit_mutable = make_orbit()
yawed_qsw = LofOffset(
    orbit_mutable.native_frame,
    LOFType.QSW,
    RotationOrder.ZYX,
    np.deg2rad(10.0),
    0.0,
    0.0,
)
orbit_mutable.set_attitude_provider(yawed_qsw)
q_yawed = orbit_mutable.get_attitude_quat(t_query, quaternion_convention="scalar_last")

print("updated provider class:", orbit_mutable.get_attitude_provider().getClass().getSimpleName())
print("QSW + 10 deg yaw first quaternion:", q_yawed[0])


updated provider class: LofOffset
QSW + 10 deg yaw first quaternion: [-0.680877  0.317498 -0.330001 -0.571578]


## 6) Rate-limited yaw steering wrapper

Use `RateLimitedYawSteeringProvider` when you want deterministic nadir-pointing plus Sun phasing with explicit yaw-rate and yaw-acceleration limits.


In [ ]:
rate_limited = RateLimitedYawSteeringProvider(
    inertial_frame="gcrf",
    reference_epoch=epoch,
    max_yaw_rate_rad_s=0.02,
    max_yaw_acceleration_rad_s2=0.002,
    kp=0.1,
    kd=0.4,
    finite_difference_step_s=0.1,
)

orbit_mutable.set_attitude_provider(rate_limited)
spin_limited = orbit_mutable.get_attitude_spin(np.linspace(0.0, 600.0, 6, dtype=np.float64))

print("rate-limited provider class:", orbit_mutable.get_attitude_provider().getClass().getSimpleName())
print("limited spin samples [rad/s]:")
print(spin_limited)


rate-limited provider class: RateLimitedYawSteeringProvider
limited spin samples [rad/s]:
[[ 0.000001 -0.001071 -0.      ]
 [ 0.001037  0.000261 -0.000215]
 [ 0.00104   0.00024   0.000195]
 [ 0.001045  0.000213  0.000226]
 [ 0.001049  0.000183  0.000253]
 [ 0.001053  0.000149  0.000276]]


## Practical guidance

- Configure attitude on the propagator when you build it whenever possible.
- Use `Orbit.set_attitude_provider(...)` when you want to swap providers after the fact.
- Query quaternions for orientation and `get_attitude_spin(...)` / `get_attitude_acceleration(...)` for body-frame kinematics.
- Use raw Orekit providers when you already know exactly what Orekit object you want.
- Use NSTK helpers and wrappers when you want a higher-level, documented shortcut.
